In [ ]:

# ============================================================
# SAHAYAKAI KARNATAKA
# Government Application Document Assistance System
# Google Colab + Gradio
# ============================================================

!pip -q install -U gradio requests beautifulsoup4 pypdf

import os
import re
import tempfile
import requests
import gradio as gr

from datetime import datetime
from bs4 import BeautifulSoup
from pypdf import PdfReader


# ============================================================
# 1. KARNATAKA DISTRICTS
# ============================================================

DISTRICTS = [
    "Bagalkot", "Ballari", "Belagavi", "Bengaluru Rural",
    "Bengaluru Urban", "Bidar", "Chamarajanagar", "Chikkaballapur",
    "Chikkamagaluru", "Chitradurga", "Dakshina Kannada",
    "Davanagere", "Dharwad", "Gadag", "Hassan", "Haveri",
    "Kalaburagi", "Kodagu", "Kolar", "Koppal", "Mandya",
    "Mysuru", "Raichur", "Ramanagara", "Shivamogga",
    "Tumakuru", "Udupi", "Uttara Kannada", "Vijayanagara",
    "Vijayapura", "Yadgir"
]


# ============================================================
# 2. DOCUMENT CHECKLIST
# ============================================================

DOCUMENTS = [
    "Aadhaar Card",
    "Ration Card",
    "Income Proof",
    "Income Certificate",
    "Address Proof",
    "Residence Certificate",
    "Caste Certificate",
    "Bank Passbook",
    "Passport-size Photograph",
    "Birth Certificate",
    "SSLC Marks Card",
    "PUC Marks Card",
    "Degree Certificate",
    "Study Certificate",
    "Transfer Certificate",
    "Disability Certificate",
    "Pension Documents",
    "Land Records",
    "RTC / Pahani",
    "Tax Paid Receipt",
    "Voter ID",
    "PAN Card",
    "Driving Licence",
    "Employment Certificate",
    "Unemployment Declaration",
    "Marriage Certificate",
    "Death Certificate",
    "Family Tree",
    "Self-declaration",
    "Application Form",
    "Other Supporting Document"
]


# ============================================================
# 3. STARTER DOCUMENT SUGGESTIONS
# These are suggestions, not official legal requirements.
# ============================================================

def suggest_documents(application):
    name = (application or "").lower()

    suggestions = [
        "Aadhaar Card",
        "Address Proof",
        "Passport-size Photograph"
    ]

    if any(word in name for word in [
        "income certificate", "income cert", "income proof"
    ]):
        suggestions += [
            "Income Proof",
            "Ration Card",
            "Self-declaration"
        ]

    elif any(word in name for word in [
        "caste certificate", "caste and income", "category certificate"
    ]):
        suggestions += [
            "Caste Certificate",
            "Income Proof",
            "Ration Card",
            "Self-declaration"
        ]

    elif any(word in name for word in [
        "residence certificate", "residential certificate",
        "domicile", "nativity"
    ]):
        suggestions += [
            "Residence Certificate",
            "Ration Card",
            "Voter ID"
        ]

    elif any(word in name for word in [
        "scholarship", "education", "student scholarship"
    ]):
        suggestions += [
            "Income Certificate",
            "Caste Certificate",
            "Bank Passbook",
            "Study Certificate",
            "SSLC Marks Card",
            "PUC Marks Card"
        ]

    elif any(word in name for word in [
        "pension", "old age pension", "widow pension"
    ]):
        suggestions += [
            "Bank Passbook",
            "Age Proof",
            "Income Proof",
            "Ration Card"
        ]

    elif any(word in name for word in [
        "farmer", "agriculture", "land", "crop"
    ]):
        suggestions += [
            "Land Records",
            "RTC / Pahani",
            "Bank Passbook",
            "Ration Card"
        ]

    elif any(word in name for word in [
        "disability", "divyang"
    ]):
        suggestions += [
            "Disability Certificate",
            "Bank Passbook",
            "Income Proof"
        ]

    elif any(word in name for word in [
        "birth certificate", "birth registration"
    ]):
        suggestions += [
            "Birth Certificate",
            "Address Proof",
            "Self-declaration"
        ]

    elif any(word in name for word in [
        "marriage certificate", "marriage registration"
    ]):
        suggestions += [
            "Marriage Certificate",
            "Address Proof",
            "Birth Certificate"
        ]

    elif any(word in name for word in [
        "death certificate", "death registration"
    ]):
        suggestions += [
            "Death Certificate",
            "Address Proof"
        ]

    elif any(word in name for word in [
        "job", "employment", "unemployment"
    ]):
        suggestions += [
            "Employment Certificate",
            "Unemployment Declaration",
            "Bank Passbook",
            "Education Certificate"
        ]

    # Keep only documents that exist in the checklist.
    # Remove duplicates while preserving order.
    valid = set(DOCUMENTS)
    result = []

    for document in suggestions:
        if document in valid and document not in result:
            result.append(document)

    return result


# ============================================================
# 4. READ AN OFFICIAL WEBPAGE
# ============================================================

def read_webpage(url):
    if not url or not url.strip():
        return "", "No source URL provided."

    url = url.strip()

    if not url.startswith(("https://", "http://")):
        return "", "Please enter a valid URL beginning with https:// or http://."

    try:
        response = requests.get(
            url,
            timeout=15,
            headers={
                "User-Agent": "Mozilla/5.0 SahayakAI-DocumentAssistant/1.0"
            }
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for element in soup([
            "script", "style", "nav", "footer", "header"
        ]):
            element.decompose()

        text = soup.get_text(separator="\n", strip=True)
        text = re.sub(r"\n{2,}", "\n", text)

        return text[:100000], "Webpage retrieved successfully."

    except Exception as error:
        return "", f"Could not read webpage: {error}"


# ============================================================
# 5. READ AN UPLOADED PDF
# ============================================================

def read_pdf(pdf_path):
    if not pdf_path:
        return "", "No PDF uploaded."

    try:
        reader = PdfReader(pdf_path)
        pages = []

        for page in reader.pages:
            pages.append(page.extract_text() or "")

        text = "\n".join(pages)
        return text[:100000], "PDF text extracted successfully."

    except Exception as error:
        return "", f"Could not read PDF: {error}"


# ============================================================
# 6. FIND POSSIBLE DOCUMENT REFERENCES IN SOURCE TEXT
# ============================================================

KEYWORDS = {
    "Aadhaar Card": ["aadhaar", "aadhar"],
    "Ration Card": ["ration card"],
    "Income Proof": ["income proof", "income details", "income document"],
    "Income Certificate": ["income certificate"],
    "Address Proof": ["address proof", "proof of address"],
    "Residence Certificate": ["residence certificate", "residential certificate"],
    "Caste Certificate": ["caste certificate", "caste proof"],
    "Bank Passbook": ["bank passbook", "bank account details", "bank statement"],
    "Passport-size Photograph": ["passport size photo", "passport-size photograph"],
    "Birth Certificate": ["birth certificate"],
    "SSLC Marks Card": ["sslc marks card", "10th marks card"],
    "PUC Marks Card": ["puc marks card", "12th marks card"],
    "Degree Certificate": ["degree certificate"],
    "Study Certificate": ["study certificate"],
    "Transfer Certificate": ["transfer certificate"],
    "Disability Certificate": ["disability certificate"],
    "Pension Documents": ["pension documents", "pension passbook"],
    "Land Records": ["land records", "land ownership"],
    "RTC / Pahani": ["rtc", "pahani"],
    "Tax Paid Receipt": ["tax paid receipt", "property tax receipt"],
    "Voter ID": ["voter id", "voter identity"],
    "PAN Card": ["pan card"],
    "Driving Licence": ["driving licence", "driving license"],
    "Employment Certificate": ["employment certificate"],
    "Unemployment Declaration": ["unemployment declaration"],
    "Marriage Certificate": ["marriage certificate"],
    "Death Certificate": ["death certificate"],
    "Family Tree": ["family tree", "family pedigree"],
    "Self-declaration": ["self declaration", "self-declaration"],
    "Application Form": ["application form"],
}


def find_documents_in_text(text):
    if not text:
        return []

    lower_text = text.lower()
    found = []

    for document, terms in KEYWORDS.items():
        if any(term in lower_text for term in terms):
            found.append(document)

    return found


# ============================================================
# 7. SUGGEST DOCUMENTS FROM APPLICATION + OFFICIAL SOURCE
# ============================================================

def update_required_documents(application, source_url, pdf_path):
    application = (application or "").strip()

    if not application:
        return (
            gr.update(choices=DOCUMENTS, value=[]),
            "Enter an application or scheme name first."
        )

    suggested = suggest_documents(application)
    messages = [
        "Application-based starter suggestions generated.",
        "These suggestions are not verified official requirements."
    ]

    source_text = ""
    source_status = ""

    if source_url and source_url.strip():
        source_text, source_status = read_webpage(source_url)
        messages.append(source_status)

    if pdf_path:
        pdf_text, pdf_status = read_pdf(pdf_path)
        source_text += "\n" + pdf_text
        messages.append(pdf_status)

    found = find_documents_in_text(source_text)

    if found:
        suggested = list(dict.fromkeys(suggested + found))
        messages.append(
            "Possible document references found in source text: "
            + ", ".join(found)
        )
        messages.append(
            "Please review these references. The system cannot determine "
            "whether every mention is a mandatory requirement."
        )
    elif source_text:
        messages.append(
            "No known document names were automatically detected. "
            "Review the official source and select requirements manually."
        )

    suggested = [doc for doc in suggested if doc in DOCUMENTS]

    return (
        gr.update(choices=DOCUMENTS, value=suggested),
        "\n".join(messages)
    )


# ============================================================
# 8. COMPARE DOCUMENTS AND GENERATE REPORT
# ============================================================

def compare_documents(
    applicant_name,
    age,
    district,
    taluk,
    village,
    occupation,
    category,
    annual_income,
    application,
    source_url,
    required_documents,
    available_documents
):
    required_documents = required_documents or []
    available_documents = available_documents or []

    applicant_name = (applicant_name or "").strip()
    application = (application or "").strip()

    if not applicant_name:
        return "Please enter the applicant's name.", None

    if not application:
        return "Please enter the application or scheme name.", None

    missing = [
        doc for doc in required_documents
        if doc not in available_documents
    ]

    extra = [
        doc for doc in available_documents
        if doc not in required_documents
    ]

    lines = [
        "=" * 65,
        "SAHAYAKAI KARNATAKA",
        "GOVERNMENT APPLICATION DOCUMENT REPORT",
        "=" * 65,
        "",
        f"Generated on: {datetime.now().strftime('%d-%m-%Y %I:%M %p')}",
        "",
        "APPLICANT DETAILS",
        "-" * 65,
        f"Applicant name: {applicant_name}",
        f"Age: {age or 'Not provided'}",
        f"District: {district or 'Not provided'}",
        f"Taluk: {taluk or 'Not provided'}",
        f"Village: {village or 'Not provided'}",
        f"Occupation: {occupation or 'Not provided'}",
        f"Category: {category or 'Not provided'}",
        f"Annual family income: {annual_income or 'Not provided'}",
        "",
        "APPLICATION DETAILS",
        "-" * 65,
        f"Application / scheme: {application}",
        f"Official source URL: {source_url or 'Not provided'}",
        "",
        "REQUIRED DOCUMENTS SELECTED",
        "-" * 65,
    ]

    if required_documents:
        lines.extend([f"[ ] {doc}" for doc in required_documents])
    else:
        lines.append("No required documents selected.")

    lines.extend([
        "",
        "DOCUMENTS AVAILABLE",
        "-" * 65
    ])

    if available_documents:
        lines.extend([f"[x] {doc}" for doc in available_documents])
    else:
        lines.append("No available documents selected.")

    lines.extend([
        "",
        "POSSIBLY MISSING DOCUMENTS",
        "-" * 65
    ])

    if missing:
        lines.extend([f"[!] {doc}" for doc in missing])
    else:
        lines.append(
            "No missing documents according to the checklist you selected."
        )

    lines.extend([
        "",
        "OTHER DOCUMENTS YOU HAVE",
        "-" * 65
    ])

    if extra:
        lines.extend([f"[-] {doc}" for doc in extra])
    else:
        lines.append("No additional documents.")

    lines.extend([
        "",
        "IMPORTANT NOTICE",
        "-" * 65,
        "This report compares only the documents selected in this app.",
        "Starter suggestions and webpage text extraction may be incomplete.",
        "A document mention on a webpage does not prove it is mandatory.",
        "Confirm current requirements with the official department or service.",
        "This report does not establish scheme eligibility or guarantee approval.",
        "",
        "=" * 65
    ])

    report = "\n".join(lines)

    status = (
        f"## Document comparison complete\n\n"
        f"- **Application:** {application}\n"
        f"- **Required documents selected:** {len(required_documents)}\n"
        f"- **Documents marked available:** {len(available_documents)}\n"
        f"- **Possibly missing:** {len(missing)}\n\n"
    )

    if missing:
        status += "**Possibly missing documents:**\n\n"
        status += "\n".join(f"- {doc}" for doc in missing)
    else:
        status += (
            "No missing items were found in your selected checklist. "
            "This does not guarantee that the application is complete."
        )

    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            suffix=".txt",
            prefix="SahayakAI_Report_",
            delete=False
        ) as file:
            file.write(report)
            report_path = file.name
    except Exception as error:
        return status + f"\n\nCould not create report: {error}", None

    return status, report_path


# ============================================================
# 9. GRADIO USER INTERFACE
# ============================================================

with gr.Blocks(title="SahayakAI Karnataka") as demo:

    gr.Markdown("""
    # SahayakAI Karnataka
    ### Government Application Document Assistance System

    Enter any government application or scheme name. Review the suggested
    required documents, tick the documents you already have, and generate
    a report showing possible missing documents.

    **Important:** Always verify requirements with the official department.
    """)

    with gr.Tab("Applicant Details"):

        applicant_name = gr.Textbox(
            label="Applicant Name",
            placeholder="Enter full name"
        )

        with gr.Row():
            age = gr.Number(
                label="Age",
                precision=0,
                minimum=0,
                maximum=120
            )

            district = gr.Dropdown(
                choices=DISTRICTS,
                label="Karnataka District",
                value="Mysuru"
            )

        with gr.Row():
            taluk = gr.Textbox(label="Taluk")
            village = gr.Textbox(label="Village / Town")

        with gr.Row():
            occupation = gr.Dropdown(
                choices=[
                    "Student", "Farmer", "Employee", "Self-employed",
                    "Unemployed", "Homemaker", "Senior Citizen", "Other"
                ],
                label="Occupation",
                value="Student"
            )

            category = gr.Dropdown(
                choices=[
                    "General", "SC", "ST", "Category 1",
                    "2A", "2B", "3A", "3B", "Other", "Prefer not to say"
                ],
                label="Category",
                value="General"
            )

        annual_income = gr.Number(
            label="Annual Family Income (₹)",
            minimum=0,
            precision=0
        )

    with gr.Tab("Application and Requirements"):

        application = gr.Textbox(
            label="Government Application / Scheme Name",
            placeholder="Example: Income Certificate, Scholarship, Pension",
            info="Enter any application name. This field is not restricted to a fixed list."
        )

        source_url = gr.Textbox(
            label="Official Service URL (optional)",
            placeholder="https://..."
        )

        pdf_upload = gr.File(
            label="Upload official requirements PDF (optional)",
            file_types=[".pdf"],
            type="filepath"
        )

        suggest_button = gr.Button(
            "Suggest Required Documents",
            variant="primary"
        )

        suggestion_status = gr.Textbox(
            label="Suggestion Status",
            lines=5,
            interactive=False
        )

        required_documents = gr.CheckboxGroup(
            choices=DOCUMENTS,
            label="Required Documents Checklist",
            info="Tick the documents required for this application. Review and edit the suggestions."
        )

        available_documents = gr.CheckboxGroup(
            choices=DOCUMENTS,
            label="Documents You Already Have",
            info="Tick all documents currently available to you."
        )

        suggest_button.click(
            fn=update_required_documents,
            inputs=[application, source_url, pdf_upload],
            outputs=[required_documents, suggestion_status]
        )

        gr.Markdown("""
        **How the checklist works**

        1. Enter your application name.
        2. Optionally add an official webpage URL or upload its PDF.
        3. Click **Suggest Required Documents**.
        4. Review the required-document checklist and change it if necessary.
        5. Tick the documents you already have.
        """)

    with gr.Tab("Missing Documents Report"):

        generate_button = gr.Button(
            "Compare Documents and Generate Report",
            variant="primary"
        )

        report_output = gr.Markdown(
            label="Comparison Result"
        )

        download_report = gr.File(
            label="Download Applicant Report"
        )

        generate_button.click(
            fn=compare_documents,
            inputs=[
                applicant_name,
                age,
                district,
                taluk,
                village,
                occupation,
                category,
                annual_income,
                application,
                source_url,
                required_documents,
                available_documents
            ],
            outputs=[report_output, download_report]
        )

    gr.Markdown("""
    ---
    **SahayakAI Karnataka | Prototype**

    This prototype does not directly connect to every Karnataka government
    department. It cannot guarantee that extracted requirements are current
    or complete, and it does not determine official eligibility.
    """)


# ============================================================
# 10. LAUNCH APP AND CREATE PUBLIC LINK
# ============================================================

demo.launch(share=True, debug=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://718a1c33012395bae7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradi